# 04 · Évaluation finale et analyse d'erreurs (français → bambara)

**But.** Donner le **score final** du modèle fine-tuné, de façon honnête, et trouver où il se trompe.

On charge l'**adaptateur LoRA** sauvegardé par le notebook 05 (sans réentraîner) et on évalue sur **tout le jeu de test** (~1 500 phrases) :

1. le score du modèle de base **et** du modèle fine-tuné, sur exactement les mêmes phrases ;
2. des **intervalles de confiance** : l'écart est-il solide, ou dû au hasard ?
3. les faiblesses : longueur des phrases, sorties vides, répétitions, copies de la source ;
4. une liste des pires phrases à annoter à la main.

| | |
|---|---|
| **Prérequis** | notebooks 01 et 05 faits (il faut `data/processed/test.tsv` et `models/nllb-fr-bam-lora/`) |
| **Matériel** | GPU NVIDIA recommandé (une quinzaine de minutes ; beaucoup plus sur processeur, où le notebook réduit l'échantillon) |
| **Ce que tu obtiens** | `results/predictions_final_test.tsv`, `results/erreurs_a_annoter.tsv`, des lignes dans `metrics.csv` |

## 1. Se repérer dans les dossiers

**Objectif.** Retrouver le dossier du projet et vérifier que l'adaptateur entraîné est bien là.

**Ce que fait la cellule.** Détecte la racine du projet, puis vérifie que `models/nllb-fr-bam-lora` et `data/processed/test.tsv` existent.

**Ce que tu dois voir.** `Racine du projet : ...`, puis le nombre de phrases de test.

> **Attention.** Si l'adaptateur est introuvable, lance d'abord le notebook 05.

In [1]:
import os
from pathlib import Path

# Ces deux réglages évitent des avertissements inutiles de Hugging Face.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Racine du projet = le dossier qui contient "data", "models", "results", "src"...
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if not (ROOT / "src").exists():
    raise RuntimeError(
        f"Dossier du projet introuvable depuis {cwd}. "
        "Lance Jupyter avec bash lancer_jupyter.sh (ou depuis le dossier du projet)."
    )

DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
for folder in (DATA_RAW, DATA_PROC, MODELS, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

print("Racine du projet :", ROOT)

import pandas as pd

ADAPTER_DIR = MODELS / "nllb-fr-bam-lora"
if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f"{ADAPTER_DIR} introuvable : lance d'abord le notebook 05.")
if not (DATA_PROC / "test.tsv").exists():
    raise FileNotFoundError("data/processed/test.tsv introuvable : lance d'abord le notebook 01.")

test = pd.read_csv(DATA_PROC / "test.tsv", sep="\t", encoding="utf-8")
print(len(test), "phrases de test")

Racine du projet : /mnt/c/Users/RobotsMali-Victus/Desktop/nlp-bambara/nlp-bambara-wsl
1499 phrases de test


## 2. Détecter le matériel

**Objectif.** Choisir le calcul sur GPU ou processeur et la taille des lots.

**Ce que fait la cellule.** Interroge PyTorch et règle la taille des lots selon la mémoire vidéo.

**Ce que tu dois voir.** Le nom de ta carte graphique, ou un message indiquant qu'on est sur processeur.

In [2]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU détecté : {gpu_name} ({vram_gb:.1f} Go de mémoire vidéo)")
else:
    device = "cpu"
    vram_gb = 0.0
    print("Aucun GPU utilisable : le calcul se fera sur le processeur (CPU), beaucoup plus lentement.")
    print("Si tu as une carte NVIDIA, relance bash installer.sh et choisis l'option NVIDIA.")
print("PyTorch :", torch.__version__, "| CUDA compilée :", torch.version.cuda)

# Taille des lots pour traduire : prudente selon la mémoire vidéo disponible.
if device == "cuda":
    TRANSLATE_BATCH = 32 if vram_gb >= 8 else 16 if vram_gb >= 4 else 8
else:
    TRANSLATE_BATCH = 8

GPU détecté : NVIDIA GeForce RTX 2050 (4.0 Go de mémoire vidéo)
PyTorch : 2.14.1+cu126 | CUDA compilée : 12.6


## 3. Réglages de l'évaluation

**Objectif.** Décider combien de phrases évaluer et si on retraduit aussi avec le modèle de base.

**Ce que fait la cellule.** `LIMIT` : tout le test sur GPU, seulement 100 phrases sur processeur. `EVAL_BASELINE = True` retraduit avec NLLB seul, sur exactement les mêmes phrases, pour une comparaison parfaitement équitable (c'est deux fois plus long).

**Ce que tu dois voir.** Rien ne s'affiche : la cellule définit seulement des réglages.

> **Attention.** Pour un essai rapide, mets `LIMIT = 200`.

In [3]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
FR, BAM = "fra_Latn", "bam_Latn"

LIMIT = None if device == "cuda" else 100   # None = tout le test
EVAL_BASELINE = True                         # True = on retraduit aussi avec NLLB seul

subset = test.head(LIMIT) if LIMIT else test
sources = subset["fr"].tolist()
refs = subset["bam"].tolist()
print(len(subset), "phrases à traduire")

1499 phrases à traduire


## 4. Fonction de traduction

**Objectif.** Disposer d'une fonction de traduction identique à celle des notebooks 03 et 05.

**Ce que fait la cellule.** Charge le tokenizer et définit `translate(model, ...)`, qui prend le modèle en paramètre (on s'en servira avec deux modèles). Mêmes réglages de décodage (4 faisceaux) que le baseline.

**Ce que tu dois voir.** Rien ne s'affiche (ou un petit message de chargement).

In [4]:
import time
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import logging as hf_logging
from peft import PeftModel

hf_logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=FR, tgt_lang=BAM)

def translate(model, texts, src=FR, tgt=BAM, batch_size=None, num_beams=4, max_new_tokens=256):
    batch_size = batch_size or TRANSLATE_BATCH
    tokenizer.src_lang = src
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outputs = [None] * len(texts)
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        batch = [texts[i] for i in idx]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            generated = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
            )
        for i, text in zip(idx, tokenizer.batch_decode(generated, skip_special_tokens=True)):
            outputs[i] = text
    return outputs

## 5. Traduire avec NLLB seul (le « avant »)

**Objectif.** Obtenir les traductions du modèle de base sur tout l'échantillon, pour la comparaison.

**Ce que fait la cellule.** Charge NLLB sans l'adaptateur, traduit toutes les phrases, puis libère la mémoire du modèle pour laisser la place au suivant.

**Ce que tu dois voir.** Une ligne `baseline : N s`. Sur un GPU de gamme moyenne, compte plusieurs minutes pour 1 500 phrases.

> **Attention.** Pendant ce temps, la cellule semble figée : c'est normal, elle travaille. L'indicateur `[*]` à gauche de la cellule disparaît quand elle a fini.

In [5]:
hyp_base = None
if EVAL_BASELINE:
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    if device == "cuda":
        model = model.half()      # fp16 d'abord sur le processeur : évite un pic de mémoire vidéo
    model = model.to(device)
    model.eval()
    t0 = time.time()
    hyp_base = translate(model, sources)
    print(f"baseline : {time.time() - t0:.0f} s")
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

baseline : 622 s


## 6. Traduire avec NLLB + LoRA (le « après »)

**Objectif.** Obtenir les traductions du modèle fine-tuné.

**Ce que fait la cellule.** Charge NLLB, y applique l'adaptateur LoRA depuis `models/nllb-fr-bam-lora`, puis **fusionne** l'adaptateur dans le modèle (`merge_and_unload`) : le modèle devient un modèle ordinaire, plus rapide à utiliser. Puis traduit les mêmes phrases.

**Ce que tu dois voir.** Une ligne `NLLB + LoRA : N s`, de durée voisine de l'étape précédente.

In [6]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = PeftModel.from_pretrained(model, str(ADAPTER_DIR)).merge_and_unload()
if device == "cuda":
    model = model.half()      # fp16 d'abord sur le processeur : évite un pic de mémoire vidéo
model = model.to(device)
model.eval()

t0 = time.time()
hyp_lora = translate(model, sources)
print(f"NLLB + LoRA : {time.time() - t0:.0f} s")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

NLLB + LoRA : 317 s


## 7. Scores avec intervalles de confiance

**Objectif.** Savoir si le gain mesuré est réel ou peut venir du hasard de l'échantillon de test.

**Ce que fait la cellule.** On *rééchantillonne* les phrases de test au hasard (avec remise) 100 fois et on recalcule les scores à chaque fois : la dispersion donne un **intervalle à 95 %** (la vraie valeur a environ 95 % de chances d'y tomber). Pour l'écart entre les deux systèmes, on utilise le même tirage pour les deux (comparaison appariée), ce qui est plus juste.

**Ce que tu dois voir.** Un tableau avec le score de chaque système suivi de `[borne basse ; borne haute]`. Si les deux intervalles ne se chevauchent pas, la différence est probablement réelle.

> **Attention.** Cette cellule peut prendre une ou deux minutes (100 recalculs × 2 systèmes × 2 métriques).

In [7]:
import numpy as np
import sacrebleu

def score(hyps, refs_):
    return {
        "bleu": sacrebleu.corpus_bleu(hyps, [refs_]).score,
        "chrf": sacrebleu.corpus_chrf(hyps, [refs_]).score,
    }

def bootstrap(systems, refs_, n_boot=100, seed=0):
    rng = np.random.default_rng(seed)
    n = len(refs_)
    draws = {name: {"bleu": [], "chrf": []} for name in systems}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r = [refs_[i] for i in idx]
        for name, hyps in systems.items():
            s = score([hyps[i] for i in idx], r)
            draws[name]["bleu"].append(s["bleu"]); draws[name]["chrf"].append(s["chrf"])
    return draws

systems = {"NLLB + LoRA": hyp_lora}
if hyp_base is not None:
    systems = {"NLLB seul": hyp_base, **systems}
draws = bootstrap(systems, refs)

rows = []
for name, hyps in systems.items():
    s = score(hyps, refs)
    lo_b, hi_b = np.percentile(draws[name]["bleu"], [2.5, 97.5])
    lo_c, hi_c = np.percentile(draws[name]["chrf"], [2.5, 97.5])
    rows.append({"système": name,
                 "BLEU": f"{s['bleu']:.2f}  [{lo_b:.1f} ; {hi_b:.1f}]",
                 "chrF": f"{s['chrf']:.2f}  [{lo_c:.1f} ; {hi_c:.1f}]"})
final = pd.DataFrame(rows)
print(f"{len(subset)} phrases de test, intervalles à 95 %")
final

1499 phrases de test, intervalles à 95 %


,système,BLEU,chrF
0,NLLB seul,6.23 [5.8 ; 6.7],27.46 [26.8 ; 28.0]
1,NLLB + LoRA,15.23 [14.6 ; 16.0],42.36 [41.6 ; 43.1]


## 8. L'écart entre les deux systèmes

**Objectif.** Quantifier le gain apporté par le fine-tuning, avec son incertitude.

**Ce que fait la cellule.** Calcule, pour chaque tirage, la différence (LoRA moins base), puis en donne la moyenne et l'intervalle à 95 %.

**Ce que tu dois voir.** Deux lignes `gain BLEU : +x.xx (IC 95 % [...])` et `gain CHRF : ...`. Si l'intervalle **ne contient pas 0**, le gain est solide.

> **Attention.** Le test vient du même domaine (presse) que l'entraînement : une partie du gain est de l'adaptation au domaine. Sur un autre type de texte, le gain serait plus faible.

In [8]:
if hyp_base is not None:
    for metric in ("bleu", "chrf"):
        diff = np.array(draws["NLLB + LoRA"][metric]) - np.array(draws["NLLB seul"][metric])
        lo, hi = np.percentile(diff, [2.5, 97.5])
        print(f"gain {metric.upper():4s} : {diff.mean():+.2f}  (IC 95 % [{lo:+.2f} ; {hi:+.2f}])")

gain BLEU : +9.00  (IC 95 % [+8.25 ; +9.78])
gain CHRF : +14.90  (IC 95 % [+13.99 ; +15.87])


## 9. Où le modèle est-il faible ? (par longueur)

**Objectif.** Voir si les erreurs se concentrent sur certaines longueurs de phrase.

**Ce que fait la cellule.** Calcule un chrF par phrase, puis la moyenne par tranche de nombre de mots, pour les deux systèmes.

**Ce que tu dois voir.** Un tableau avec les colonnes `chrf_base`, `chrf_lora` et `n` (nombre de phrases par tranche).

In [9]:
from sacrebleu.metrics import CHRF

chrf = CHRF()
view = subset[["fr", "bam"]].copy().reset_index(drop=True)
view["lora"] = hyp_lora
view["chrf_lora"] = [chrf.sentence_score(h, [r]).score for h, r in zip(hyp_lora, refs)]
if hyp_base is not None:
    view["baseline"] = hyp_base
    view["chrf_base"] = [chrf.sentence_score(h, [r]).score for h, r in zip(hyp_base, refs)]

view["mots_fr"] = view["fr"].str.split().str.len()
view["tranche"] = pd.cut(view["mots_fr"], [0, 5, 10, 20, 35, 1000], labels=["≤5", "6-10", "11-20", "21-35", ">35"])
cols = [c for c in ("chrf_base", "chrf_lora") if c in view]
view.groupby("tranche", observed=True)[cols].mean().round(1).assign(n=view.groupby("tranche", observed=True).size())

,chrf_base,chrf_lora,n
tranche,,,
≤5,21.5,38.4,130
6-10,26.6,37.9,203
11-20,28.0,40.2,497
21-35,28.9,43.4,445
>35,29.8,44.8,224


## 10. Détecter les problèmes typiques

**Objectif.** Compter automatiquement les défauts classiques d'un traducteur neuronal.

**Ce que fait la cellule.** Repère, dans les sorties du modèle fine-tuné : les sorties vides, les copies de la phrase source, les répétitions de séquences de 3 mots, et les sorties beaucoup trop courtes ou trop longues par rapport à la référence.

**Ce que tu dois voir.** Un tableau avec le nombre et le pourcentage de phrases concernées par chaque défaut. Regarde lesquels dominent : une sortie « beaucoup trop courte » signale que le modèle s'arrête trop tôt ou résume la phrase.

In [10]:
def repeated_ngram(text, n=3):
    w = text.split()
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return len(grams) != len(set(grams))

len_ref = view["bam"].str.split().str.len().clip(lower=1)
len_hyp = view["lora"].str.split().str.len()
ratio = len_hyp / len_ref

flags = pd.DataFrame({
    "sortie vide": view["lora"].str.strip() == "",
    "copie de la source": view["lora"].str.strip() == view["fr"].str.strip(),
    "répétition (3-grammes)": view["lora"].map(repeated_ngram),
    "beaucoup trop courte (< 50 % de la réf.)": ratio < 0.5,
    "beaucoup trop longue (> 200 % de la réf.)": ratio > 2.0,
})
flags.sum().to_frame("phrases").assign(pourcentage=(flags.mean() * 100).round(1))

,phrases,pourcentage
sortie vide,0,0.0
copie de la source,1,0.1
répétition (3-grammes),76,5.1
beaucoup trop courte (< 50 % de la réf.),24,1.6
beaucoup trop longue (> 200 % de la réf.),15,1.0


## 11. Les pires phrases, à relire à la main

**Objectif.** Trouver les erreurs réelles du modèle, celles qu'un chiffre ne dit pas.

**Ce que fait la cellule.** Affiche les 15 phrases au plus mauvais score chrF, avec la référence et la sortie du modèle.

**Ce que tu dois voir.** Des phrases où la sortie est très différente de la référence.

> **Attention.** Pour chaque phrase, décide : **le modèle a-t-il tort, ou la référence est-elle une traduction très libre ?** Les deux arrivent souvent. Tu es la personne la mieux placée pour trancher.

In [11]:
pd.set_option("display.max_colwidth", 160)
worst = view.sort_values("chrf_lora").head(15)
for _, r in worst.iterrows():
    print(f"chrF {r['chrf_lora']:.0f}")
    print("FR       :", r["fr"])
    print("REF  bam :", r["bam"])
    print("LoRA     :", r["lora"])
    print()

chrF 1
FR       : C’est faux à 100%.
REF  bam : Nkalon jugu don.
LoRA     : O tɛ tiɲɛ ye 100%.

chrF 3
FR       : Par ailleurs.
REF  bam : Fan wɛrɛ fɛ,
LoRA     : O kɔ,

chrF 3
FR       : Du vril !
REF  bam : Ko tɔnɔmininen dɛrɛsi !
LoRA     : A ka ɲi !

chrF 4
FR       : La preuve.
REF  bam : A dalilu,
LoRA     : Dajira.

chrF 5
FR       : A savoir.
REF  bam : K’a dɔn k’a fɔ,
LoRA     : O kɔrɔ.

chrF 5
FR       : Comme des godillots enivrés.
REF  bam : I ko mɔgɔ sulenw, minnu t’u yɛrɛ bolo.
LoRA     : I na fɔ dɔrɔtɔw.

chrF 5
FR       : TRANSMISSION ET DENONCIATION DE FAITS PAR LE VERIFICATEUR GENERAL AU PRESIDENT DE LA SECTION DES COMPTES DE LA COUR SUPREME ET AU PROCUREUR DE LA REPUBLIQUE PRES LE TRIBUNAL DE GRANDE INSTANCE DE LA COMMUNE III DU DISTRICT DE BAMAKO, CHARGE DU POLE ECONOMIQUE ET FINANCIER ET RELATIVEMENT.
REF  bam : WALEYAW LATƐMƐNI AN’U KOFƆLI KIRITIGƐBULONBA JATEBƆW BOLOFARA ƝƐMƆGƆBA MA SƐGƐSƐGƐLIKƐLABA FƐ ANI JAMANA LAFASASARIYATIGI MIN BƐ BAMAKO DISITIRIKI KOMINI I

## 12. Sauvegarder

**Objectif.** Garder toutes les traductions, un fichier d'annotation et les scores finaux.

**Ce que fait la cellule.** Écrit `results/predictions_final_test.tsv` (toutes les traductions) et `results/erreurs_a_annoter.tsv` (les 60 pires phrases avec deux colonnes vides, `type_erreur` et `commentaire`, que tu remplis à la main dans un tableur (LibreOffice ou Excel)), puis ajoute les scores finaux à `metrics.csv`.

**Ce que tu dois voir.** Trois lignes qui listent les fichiers écrits.

In [12]:
from datetime import datetime

view.drop(columns=["tranche"]).to_csv(RESULTS / "predictions_final_test.tsv", sep="\t", index=False, encoding="utf-8")

review = view.sort_values("chrf_lora").head(60)[["fr", "bam", "lora", "chrf_lora"]].copy()
review["type_erreur"] = ""
review["commentaire"] = ""
review.to_csv(RESULTS / "erreurs_a_annoter.tsv", sep="\t", index=False, encoding="utf-8")

metrics_path = RESULTS / "metrics.csv"
new = not metrics_path.exists() or metrics_path.stat().st_size == 0
with metrics_path.open("a", encoding="utf-8", newline="") as f:
    if new:
        f.write("date,experiment,model,direction,n_sentences,bleu,chrf\n")
    now = datetime.now().isoformat(timespec="seconds")
    for exp, hyps in (("baseline-final", hyp_base), ("lora-final", hyp_lora)):
        if hyps is None:
            continue
        s = score(hyps, refs)
        f.write(f"{now},{exp},{MODEL_NAME},fr-bam,{len(hyps)},{s['bleu']:.2f},{s['chrf']:.2f}\n")

print("Fichiers écrits dans", RESULTS)
print(" - predictions_final_test.tsv  (toutes les traductions)")
print(" - erreurs_a_annoter.tsv      (60 pires phrases, à remplir)")
print(" - metrics.csv                (scores ajoutés)")

Fichiers écrits dans /mnt/c/Users/RobotsMali-Victus/Desktop/nlp-bambara/nlp-bambara-wsl/results
 - predictions_final_test.tsv  (toutes les traductions)
 - erreurs_a_annoter.tsv      (60 pires phrases, à remplir)
 - metrics.csv                (scores ajoutés)


## Suite

- Ouvre `results/erreurs_a_annoter.tsv` dans un tableur (LibreOffice ou Excel) et remplis les colonnes `type_erreur` et `commentaire` : c'est la matière de ton analyse d'erreurs.
- Selon ce que tu y vois, décide du prochain entraînement : plus d'époques, un modèle plus grand, ou plus de données (par exemple tes propres paires).
- **Notebook 06** : une interface web pour utiliser le traducteur.

*Données : MAFAND-MT, Masakhane, CC BY-NC-4.0. Modèle : NLLB-200, Meta.*